# ✅ Solution 3: Statistik Deskriptif & Eksplorasi DataFrame

**Approach**: groupby + aggregasi + visualisasi bar chart  
**Key Insight**: Pertanyaan bisnis bisa dijawab dari statistik deskriptif saja — tanpa model ML.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
np.random.seed(42)

# Regenerasi dataset yang sama
n = 200
cabang = np.random.choice(['Jakarta','Bandung','Surabaya'], size=n, p=[0.5,0.3,0.2])
kategori = np.random.choice(['Elektronik','Fashion','Makanan','Perabot'], size=n, p=[0.3,0.35,0.25,0.1])
harga_base = {'Elektronik':1500000,'Fashion':250000,'Makanan':75000,'Perabot':500000}
harga = np.array([harga_base[k]*(0.7+np.random.random()*0.8) for k in kategori])
jumlah = np.random.randint(1,6,size=n)
total = harga * jumlah

df_penjualan = pd.DataFrame({
    'cabang': cabang, 'kategori': kategori,
    'harga_satuan': harga.round(0), 'jumlah_terjual': jumlah,
    'total_penjualan': total.round(0),
    'metode_bayar': np.random.choice(['Cash','Transfer','Kartu Kredit','QRIS'],size=n),
})
print(f'Dataset: {df_penjualan.shape}'); df_penjualan.head(3)

In [ ]:
# ============================================================
# SOLUSI TODO 1: Ringkasan statistik + jawaban 3 pertanyaan
# ============================================================
print('=== .info() ==='); df_penjualan.info(); print()
print('=== .describe() ==='); display(df_penjualan.describe().round(0)); print()

# a. Rata-rata total penjualan
rata_total = df_penjualan['total_penjualan'].mean()
print(f'a. Rata-rata total penjualan: Rp {rata_total:,.0f}')

# b. Jumlah transaksi per cabang
print('b. Jumlah transaksi per cabang:')
print(df_penjualan['cabang'].value_counts())

# c. Metode bayar paling sering
top_bayar = df_penjualan['metode_bayar'].value_counts().index[0]
print(f'c. Metode bayar terpopuler: {top_bayar}')

In [ ]:
# ============================================================
# SOLUSI TODO 2: Analisis per Cabang
# ============================================================
agg_cabang = df_penjualan.groupby('cabang').agg(
    total_pendapatan=('total_penjualan','sum'),
    rata_total=('total_penjualan','mean'),
    rata_jumlah=('jumlah_terjual','mean'),
    jumlah_transaksi=('total_penjualan','count')
).round(0)

print('Analisis per Cabang:'); display(agg_cabang)

print()
print(f"a. Cabang total penjualan tertinggi: {agg_cabang['total_pendapatan'].idxmax()}")
print(f"b. Cabang rata-rata jumlah terjual tertinggi: {agg_cabang['rata_jumlah'].idxmax()}")
print('c. Total pendapatan per cabang:')
for cab, row in agg_cabang.iterrows():
    print(f'   {cab}: Rp {row["total_pendapatan"]:,.0f}')

In [ ]:
# ============================================================
# SOLUSI TODO 3: Visualisasi
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot kiri: total penjualan per cabang
total_per_cabang = df_penjualan.groupby('cabang')['total_penjualan'].sum() / 1e6
total_per_cabang.sort_values(ascending=False).plot(
    kind='bar', ax=axes[0], color=['#2196F3','#4CAF50','#FF9800'], edgecolor='black')
axes[0].set_title('Total Penjualan per Cabang', fontweight='bold')
axes[0].set_xlabel('Cabang')
axes[0].set_ylabel('Total Penjualan (Juta Rp)')
axes[0].tick_params(axis='x', rotation=0)
for bar in axes[0].patches:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{bar.get_height():.0f}M', ha='center', fontsize=10)

# Plot kanan: jumlah transaksi per kategori
df_penjualan['kategori'].value_counts().plot(
    kind='bar', ax=axes[1], color='#9C27B0', edgecolor='black')
axes[1].set_title('Jumlah Transaksi per Kategori', fontweight='bold')
axes[1].set_xlabel('Kategori')
axes[1].set_ylabel('Jumlah Transaksi')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SOLUSI TODO 4: Star Performer (cabang + kategori terbaik)
# ============================================================
agg_2 = df_penjualan.groupby(['cabang','kategori'])['total_penjualan'].mean()
star = agg_2.idxmax()
star_value = agg_2.max()

print(f'Star Performer: {star[0]} - {star[1]}')
print(f'Rata-rata total penjualan: Rp {star_value:,.0f}')
print()
print('Top 5 kombinasi:')
print(agg_2.sort_values(ascending=False).head(5).apply(lambda x: f'Rp {x:,.0f}'))

## 📌 Takeaways

- `groupby('kolom').agg({'kol': 'func'})` → agregasi per grup, lebih fleksibel dari `groupby().mean()`
- `idxmax()` → kembalikan index (label) dari nilai maksimum
- Visualisasi sederhana (bar chart) sudah cukup untuk menjawab banyak pertanyaan bisnis
- **Median vs Mean**: Untuk data penjualan yang mungkin skewed, median lebih robust dari mean
- Groupby dua kolom → `groupby(['kol1','kol2'])` menghasilkan MultiIndex